# Sandbox Analysis: Natural Language Bioeconomy Exploration

Welcome to the AI-driven data sandbox for the `ca-biositing` project. This notebook uses **PandasAI 3.0+** and the **CBORG API** (Gemini models) to allow you to explore geospatial bioeconomy data using natural language.

### Capabilities:
- **Natural Language Querying**: Ask questions about the data in plain English.
- **Interactive Visualizations**: Generate Plotly charts for zooming and hovering.
- **Local DB Integration**: Queries the live `ca_biositing` PostgreSQL instance.

## 1. Setup and Environment

We initialize our connection to the CBORG LLM gateway and apply a **Global Patch** to PandasAI to ensure all interactive charts render directly in the notebook.

In [2]:
import os
import pandas as pd
import requests
import plotly.io as pio
import plotly.graph_objects as go
from dotenv import load_dotenv
from pandasai import Agent
from pandasai.llm.base import LLM
import pandasai.core.response.base as response_base
from sqlalchemy import create_engine
from IPython.display import HTML, display

# Set Plotly as default renderer for Jupyter
pio.renderers.default = 'notebook_connected'

# Load environment variables
load_dotenv()

class CBORGLLM(LLM):
    """Custom LLM class for CBORG (OpenAI-compatible) gateway in PandasAI 3.0"""
    def __init__(self, api_token, api_base="https://api.cborg.lbl.gov/v1", model="gemini-2.0-flash"):
        super().__init__()
        self.api_token = api_token
        self.api_base = api_base
        self.model = model

    def call(self, instruction, context=None):
        prompt = instruction.to_string()
        headers = {
            "Authorization": f"Bearer {self.api_token}",
            "Content-Type": "application/json"
        }
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0
        }
        response = requests.post(f"{self.api_base}/chat/completions", headers=headers, json=payload)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]

    @property
    def type(self):
        return "cborg"

# --- GLOBAL PATCH FOR INTERACTIVE RENDERING ---
def patched_repr_html(self):
    """Ensure any response containing an HTML path or Plotly object renders in the notebook"""
    try:
        # 1. Handle direct Figure objects
        if hasattr(self.value, 'to_html'):
            return self.value.to_html(include_plotlyjs='cdn', full_html=False)
        
        # 2. Handle HTML file paths
        if isinstance(self.value, str) and self.value.endswith('.html'):
            # Extract filename just in case it's a relative path mismatch
            fname = os.path.basename(self.value)
            
            # Search paths: direct, relative to CWD, relative to project root
            paths_to_try = [
                self.value,
                os.path.join(os.getcwd(), self.value),
                os.path.abspath(os.path.join(os.getcwd(), "..", "..", self.value)),
                os.path.abspath(os.path.join(os.getcwd(), "..", "..", "exports", "charts", fname))
            ]
            
            for path in paths_to_try:
                if os.path.exists(path):
                    with open(path, 'r') as f:
                        return f.read()
    except Exception as e:
        return f"<p style='color:red'>Rendering Error: {e}</p>"
    
    return None

# Apply the patch to the base response class
response_base.BaseResponse._repr_html_ = patched_repr_html
print("PandasAI patched globally for interactive Plotly rendering.")

api_key = os.getenv("CBORG_API_KEY")
api_url = os.getenv("CBORG_API_URL", "https://api.cborg.lbl.gov/v1")
model_name = os.getenv("CBORG_MODEL", "gemini-3-flash")

if not api_key:
    raise ValueError("CBORG_API_KEY not found. Please check your .env file in analysis/ai_exploration/.")

llm = CBORGLLM(api_token=api_key, api_base=api_url, model=model_name)
print(f"Using model: {model_name} via {api_url}")

PandasAI patched globally for interactive Plotly rendering.
Using model: gemini-3-flash via https://api.cborg.lbl.gov/v1


## 2. Database Connectivity

Connect to the local PostgreSQL database.

In [3]:
DB_USER = os.getenv("DB_USER", "biocirv_user")
DB_PASS = os.getenv("DB_PASSWORD", "biocirv_dev_password")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "biocirv_db")
DB_SCHEMA = os.getenv("DB_SCHEMA", "ca_biositing")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)
print(f"Connected to {DB_NAME}")

Connected to biocirv_db


## 3. Loading Data and Initializing Agent

We load all relevant views into the agent.

In [4]:
views = ["analysis_data_view", "usda_census_view", "analysis_average_view"]
loaded_dfs = {}
for v in views:
    try:
        loaded_dfs[v] = pd.read_sql(f"SELECT * FROM {DB_SCHEMA}.{v}", engine)
        print(f"- Loaded {v}: {len(loaded_dfs[v])} rows")
    except Exception as e:
        print(f"- Skipping {v}: {e}")

# Initialize the Agent with all loaded data
agent = Agent(
    list(loaded_dfs.values()), 
    config={
        "llm": llm,
        "verbose": True
    }
)

print(f"\nAgent initialized with {len(loaded_dfs)} datasets.")

- Loaded analysis_data_view: 8371 rows
- Loaded usda_census_view: 1419 rows
- Loaded analysis_average_view: 773 rows

Agent initialized with 3 datasets.


/var/folders/2l/qpqn5_6578z142wxn32lbtw00000gn/T/ipykernel_98625/1629183325.py:11: DeprecationWarning: The 'config' parameter is deprecated and will be removed in a future version. Please use the global configuration instead.
  agent = Agent(


## 4. Interactive Queries

Ask for an interactive chart below. The global patch ensures it renders as interactive HTML in the cell output.

### A. Interactive Visualization
Visualize the energy content distribution with an interactive Plotly bar chart.

In [ ]:
agent.chat("Generate an interactive Plotly bar chart showing the distribution of energy content across different feedstock types from the analysis data.")

In [6]:
agent.chat("Plot xylan vs glucan for all resources in the analysis_average view")

2026-03-27 12:44:05 [INFO] Question: Plot xylan vs glucan for all resources in the analysis_average view
2026-03-27 12:44:05 [INFO] Running PandasAI with cborg LLM...
2026-03-27 12:44:05 [INFO] Prompt ID: 6cb8bb2c-4d58-497c-b3b6-736dfc270c8e
2026-03-27 12:44:05 [INFO] Generating new code...
2026-03-27 12:44:05 [INFO] Using Prompt: <tables>

<table dialect="duckdb" table_name="table_c3d589fa268857ad142c9f3c84f8aaf5" columns="[{"name": "id", "type": "integer", "description": null, "expression": null, "alias": null}, {"name": "record_id", "type": "string", "description": null, "expression": null, "alias": null}, {"name": "record_type", "type": "string", "description": null, "expression": null, "alias": null}, {"name": "resource_id", "type": "float", "description": null, "expression": null, "alias": null}, {"name": "resource", "type": "string", "description": null, "expression": null, "alias": null}, {"name": "geoid", "type": "string", "description": null, "expression": null, "alias": null

ChartResponse(type='chart', value='exports/charts/temp_chart_572072f3-767f-4643-9900-d3160faeda2b.png')

### B. Scatter Plot
Visualize relationship between two variables from the average view.

In [5]:
agent.chat("Plot xylan vs glucan for all resources in the analysis_average view using an interactive Plotly scatter plot.")

2026-03-27 12:41:51 [INFO] Question: Plot xylan vs glucan for all resources in the analysis_average view using an interactive Plotly scatter plot.
2026-03-27 12:41:52 [INFO] Running PandasAI with cborg LLM...
2026-03-27 12:41:52 [INFO] Prompt ID: 76f5f9d6-3812-4e73-9e76-4b96446f8586
2026-03-27 12:41:52 [INFO] Generating new code...
2026-03-27 12:41:52 [INFO] Using Prompt: <tables>

<table dialect="duckdb" table_name="table_c3d589fa268857ad142c9f3c84f8aaf5" columns="[{"name": "id", "type": "integer", "description": null, "expression": null, "alias": null}, {"name": "record_id", "type": "string", "description": null, "expression": null, "alias": null}, {"name": "record_type", "type": "string", "description": null, "expression": null, "alias": null}, {"name": "resource_id", "type": "float", "description": null, "expression": null, "alias": null}, {"name": "resource", "type": "string", "description": null, "expression": null, "alias": null}, {"name": "geoid", "type": "string", "description

2026-03-27 12:43:10 [INFO] Retrying execution (2/3)...
2026-03-27 12:43:10 [INFO] Execution failed with error: Traceback (most recent call last):
  File "/Users/pjsmitty301/ca-biositing/analysis/ai_exploration/.pixi/envs/default/lib/python3.11/site-packages/pandasai/core/code_execution/code_executor.py", line 29, in execute
    exec(code, self._environment)
  File "<string>", line 14, in <module>
  File "/Users/pjsmitty301/ca-biositing/analysis/ai_exploration/.pixi/envs/default/lib/python3.11/site-packages/plotly/basedatatypes.py", line 3420, in show
    return pio.show(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pjsmitty301/ca-biositing/analysis/ai_exploration/.pixi/envs/default/lib/python3.11/site-packages/plotly/io/_renderers.py", line 415, in show
    raise ValueError(
ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

The above exception was the direct cause of the following exception:

Traceback (most recent call

2026-03-27 12:43:44 [INFO] Retrying execution (3/3)...
2026-03-27 12:43:44 [INFO] Execution failed with error: Traceback (most recent call last):
  File "/Users/pjsmitty301/ca-biositing/analysis/ai_exploration/.pixi/envs/default/lib/python3.11/site-packages/pandasai/core/code_execution/code_executor.py", line 29, in execute
    exec(code, self._environment)
  File "<string>", line 17, in <module>
  File "/Users/pjsmitty301/ca-biositing/analysis/ai_exploration/.pixi/envs/default/lib/python3.11/site-packages/plotly/basedatatypes.py", line 3420, in show
    return pio.show(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pjsmitty301/ca-biositing/analysis/ai_exploration/.pixi/envs/default/lib/python3.11/site-packages/plotly/io/_renderers.py", line 415, in show
    raise ValueError(
ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

The above exception was the direct cause of the following exception:

Traceback (most recent call

KeyboardInterrupt: 

### C. Multi-DataFrame Analysis
Correlate production data with USDA census data.

In [ ]:
agent.chat("Summarize how the county-level production in the analysis data relates to the census data in the USDA view.")